<a href="https://colab.research.google.com/github/zbovaird/Wolfram-Guardrails/blob/main/notebooks/colab_parser_qlora_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Wolfram Guardrails Parser QLoRA Fine-Tune (Colab GPU)

Fine-tune **Qwen2.5-3B-Instruct** as a semantic JSON parser, then compare the tuned model against the base Ollama parser on blind holdout prompts.

**Recommended runtime:** `Runtime -> Change runtime type -> A100 GPU` (also works on L4/A10; T4 fallback settings included).

```text
English prompt -> semantic JSON parser -> Wolfram guardrails (Python mirror in Colab) -> ALLOW/BLOCK/REVIEW
```

Promotion gate on Cycle 6 holdout: **zero false allows**.

Dataset paths in this repo:
- `datasets/finetune/splits/train.chat.jsonl` (1,231 rows)
- `datasets/finetune/splits/validation.chat.jsonl` (155 rows)
- `datasets/finetune/splits/test.chat.jsonl` (154 rows)
- `datasets/cycle6_fresh_blind_holdout_eval_prompts.jsonl` (180 rows)


## 1. Runtime check

Use an A100 if available. The next cell auto-tunes batch size and sequence length from the detected GPU.


In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print('Total VRAM (GiB):', round(props.total_memory / (1024 ** 3), 1))
else:
    raise RuntimeError('GPU runtime required. Use Runtime -> Change runtime type -> GPU.')


## 2. Configuration

Set `REPO_URL` to your Wolfram Guardrails GitHub repo before running. If the repo is private, use a [personal access token](https://github.com/settings/tokens) in the clone URL or upload a zip to `/content/wolfram-guardrails`.


In [ ]:
import os
from pathlib import Path

# Update if needed. Example private clone:
# REPO_URL = 'https://<TOKEN>@github.com/<user>/wolfram-guardrails.git'
REPO_URL = os.environ.get('WOLFRAM_GUARDRAILS_REPO', 'https://github.com/zbovaird/Wolfram-Guardrails.git')
REPO_BRANCH = os.environ.get('WOLFRAM_GUARDRAILS_BRANCH', 'main')
REPO_DIR = Path('/content/wolfram-guardrails')

HF_BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
OLLAMA_BASE_MODEL = 'qwen2.5:3b'
RUN_NAME = 'wolfram-guardrails-qwen25-3b-parser-v1'

TRAIN_FILE = REPO_DIR / 'datasets/finetune/splits/train.chat.jsonl'
VALIDATION_FILE = REPO_DIR / 'datasets/finetune/splits/validation.chat.jsonl'
TEST_FILE = REPO_DIR / 'datasets/finetune/splits/test.chat.jsonl'
CYCLE6_HOLDOUT = REPO_DIR / 'datasets/cycle6_fresh_blind_holdout_eval_prompts.jsonl'

OUTPUT_ROOT = REPO_DIR / 'results/finetune/colab_qwen25_3b'
ADAPTER_DIR = OUTPUT_ROOT / 'adapter'
MERGED_DIR = OUTPUT_ROOT / 'merged_hf'
COMPARE_DIR = REPO_DIR / 'results/parser_model_comparison/colab_cycle6'

import torch
_gpu = torch.cuda.get_device_name(0).upper() if torch.cuda.is_available() else ''
if 'A100' in _gpu:
    GPU_PROFILE = 'a100'
    MAX_SEQ_LENGTH = 2048
    PER_DEVICE_BATCH_SIZE = 4
    GRADIENT_ACCUMULATION_STEPS = 4
    USE_BF16 = True
elif any(tag in _gpu for tag in ('L4', 'A10', 'A40')):
    GPU_PROFILE = 'l4_a10'
    MAX_SEQ_LENGTH = 1536
    PER_DEVICE_BATCH_SIZE = 2
    GRADIENT_ACCUMULATION_STEPS = 4
    USE_BF16 = True
else:
    GPU_PROFILE = 't4_fallback'
    MAX_SEQ_LENGTH = 1024
    PER_DEVICE_BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 8
    USE_BF16 = False

NUM_TRAIN_EPOCHS = 3
LEARNING_RATE = 2e-4

print('Repo:', REPO_URL)
print('Branch:', REPO_BRANCH)
print('GPU profile:', GPU_PROFILE)
print('HF base:', HF_BASE_MODEL)
print('Batch size:', PER_DEVICE_BATCH_SIZE, 'x grad accum', GRADIENT_ACCUMULATION_STEPS)
print('Max seq length:', MAX_SEQ_LENGTH)
print('Train file:', TRAIN_FILE)


## 3. Install dependencies


In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd git
!pip -q install -U pip
!pip -q install -U \
  'transformers>=4.45.0' \
  'datasets>=2.20.0' \
  'accelerate>=0.33.0' \
  'peft>=0.12.0' \
  'trl>=0.9.6' \
  'bitsandbytes>=0.43.3' \
  'httpx>=0.27' \
  'pydantic>=2.7' \
  sentencepiece \
  'protobuf>=5.29.1,<6.0.0' \
  safetensors
!curl -fsSL https://ollama.com/install.sh | sh
!python -m pip check || true


## 4. Clone repository and verify datasets


In [ ]:
import subprocess
from pathlib import Path

import shutil

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone_cmd = ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)]
print('Running:', ' '.join(clone_cmd))
subprocess.run(clone_cmd, check=True)

required = [TRAIN_FILE, VALIDATION_FILE, TEST_FILE, CYCLE6_HOLDOUT]
missing = [str(p.relative_to(REPO_DIR)) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing dataset files after clone: ' + ', '.join(missing))

for path in required:
    lines = sum(1 for ln in path.read_text().splitlines() if ln.strip())
    print(f'{path.relative_to(REPO_DIR)}: {lines} rows')

import os
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())


## 5. Start Ollama and pull base model

Ollama serves the **base** parser for comparison after fine-tuning.


In [ ]:
import subprocess, time

ollama_log = open('/content/ollama.log', 'w')
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
time.sleep(8)
!ollama pull qwen2.5:3b
!ollama list


## 6. Load chat fine-tune splits


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(HF_BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

raw = load_dataset('json', data_files={
    'train': str(TRAIN_FILE),
    'validation': str(VALIDATION_FILE),
    'test': str(TEST_FILE),
})

def render_chat(row):
    return {'text': tokenizer.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=False)}

dataset = raw.map(render_chat, remove_columns=raw['train'].column_names)
print(dataset)
print(dataset['train'][0]['text'][:800])


## 7. Train QLoRA adapter

Settings scale with GPU profile. Effective batch size stays near 16 via gradient accumulation.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    HF_BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

train_kwargs = {
    'output_dir': str(OUTPUT_ROOT / 'trainer'),
    'run_name': RUN_NAME,
    'dataset_text_field': 'text',
    'max_seq_length': MAX_SEQ_LENGTH,
    'packing': False,
    'num_train_epochs': NUM_TRAIN_EPOCHS,
    'per_device_train_batch_size': PER_DEVICE_BATCH_SIZE,
    'per_device_eval_batch_size': max(1, PER_DEVICE_BATCH_SIZE // 2),
    'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
    'learning_rate': LEARNING_RATE,
    'lr_scheduler_type': 'cosine',
    'warmup_ratio': 0.05,
    'logging_steps': 10,
    'eval_strategy': 'steps',
    'eval_steps': 50,
    'save_steps': 50,
    'save_total_limit': 2,
    'report_to': 'none',
    'optim': 'paged_adamw_8bit',
}
if USE_BF16:
    train_kwargs['bf16'] = True
else:
    train_kwargs['fp16'] = True

training_args = SFTConfig(**train_kwargs)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    peft_config=lora_config,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print('Saved adapter to', ADAPTER_DIR)


## 8. Adapter smoke test


In [ ]:
import json, torch

sample = json.loads(CYCLE6_HOLDOUT.read_text().splitlines()[0])
messages = [
    {'role': 'system', 'content': 'You are a semantic parser for Wolfram Guardrails. Return strict JSON only.'},
    {'role': 'user', 'content': sample['prompt']},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to(trainer.model.device)
with torch.no_grad():
    outputs = trainer.model.generate(**inputs, max_new_tokens=384, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print('Prompt:', sample['prompt'])
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))


## 9. Merge adapter into full HF checkpoint


In [ ]:
import gc, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

del trainer, model
gc.collect()
torch.cuda.empty_cache()

merge_dtype = torch.bfloat16 if USE_BF16 else torch.float16
base_for_merge = AutoModelForCausalLM.from_pretrained(
    HF_BASE_MODEL,
    torch_dtype=merge_dtype,
    device_map='auto',
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base_for_merge, str(ADAPTER_DIR)).merge_and_unload()
MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(str(MERGED_DIR), safe_serialization=True, max_shard_size='2GB')
tokenizer.save_pretrained(str(MERGED_DIR))
print('Saved merged model to', MERGED_DIR)


## 10. Compare base Ollama vs fine-tuned parser

End-to-end metric: `prompt -> semantic JSON -> guardrails/policy.py -> decision`.

Start with `--limit 30` for a quick check, then run the full Cycle 6 holdout (180 prompts).


In [ ]:
!python examples/compare_colab_qlora_parser.py \
  --dataset datasets/cycle6_fresh_blind_holdout_eval_prompts.jsonl \
  --output-dir results/parser_model_comparison/colab_cycle6_smoke \
  --base-ollama-model qwen2.5:3b \
  --candidate-model results/finetune/colab_qwen25_3b/merged_hf \
  --limit 30

!cat results/parser_model_comparison/colab_cycle6_smoke/latest/summary.json


In [ ]:
# Full Cycle 6 holdout (~180 prompts). Allow several minutes on A100.
!python examples/compare_colab_qlora_parser.py \
  --dataset datasets/cycle6_fresh_blind_holdout_eval_prompts.jsonl \
  --output-dir results/parser_model_comparison/colab_cycle6 \
  --base-ollama-model qwen2.5:3b \
  --candidate-model results/finetune/colab_qwen25_3b/merged_hf

!cat results/parser_model_comparison/colab_cycle6/latest/summary.json


## 11. Optional: export merged model to Ollama


In [ ]:
# Optional. Run after merged_hf exists.
import os, subprocess
from pathlib import Path

os.chdir('/content')
if not Path('/content/llama.cpp').exists():
    subprocess.run(['git', 'clone', 'https://github.com/ggerganov/llama.cpp.git', '/content/llama.cpp'], check=True)
subprocess.run(['pip', 'install', '-q', '-r', '/content/llama.cpp/requirements.txt'], check=True)
gguf_f16 = OUTPUT_ROOT / f'{RUN_NAME}.f16.gguf'
subprocess.run([
    'python', '/content/llama.cpp/convert_hf_to_gguf.py', str(MERGED_DIR),
    '--outfile', str(gguf_f16), '--outtype', 'f16',
], check=True)
modelfile = OUTPUT_ROOT / 'Modelfile'
modelfile.write_text(f'''FROM {gguf_f16}
PARAMETER temperature 0
SYSTEM You are a semantic parser for Wolfram Guardrails. Return strict JSON only.
''')
subprocess.run(['ollama', 'create', RUN_NAME, '-f', str(modelfile)], check=True)
!ollama list
os.chdir(REPO_DIR)


## 12. Download artifacts


In [ ]:
import shutil
from google.colab import files

os.chdir(REPO_DIR)
archive_base = '/content/wolfram_guardrails_qwen25_3b_parser_v1_artifacts'
shutil.make_archive(archive_base, 'zip', root_dir=str(OUTPUT_ROOT))
print('Archive:', archive_base + '.zip')
print('Comparison summary:', COMPARE_DIR / 'latest/summary.json')
# files.download(archive_base + '.zip')  # uncomment to auto-download


## 13. Local Wolfram-backed follow-up

After importing the tuned model into local Ollama, re-run comparison with Wolfram Engine installed:

```bash
python examples/compare_colab_qlora_parser.py \
  --dataset datasets/cycle6_fresh_blind_holdout_eval_prompts.jsonl \
  --output-dir results/parser_model_comparison/local_cycle6 \
  --base-ollama-model qwen2.5:3b \
  --candidate-model <path-to-merged_hf-or-ollama-model>
```

Colab uses the Python policy mirror (`guardrails/policy.py`). Local Wolfram Engine is the production promotion gate.
